# MT06 — Proximal Policy Optimization (PPO)

### Lab Description

The MT01–MT05 labs used *scripted* experts, *behavior cloning*, and a pretrained *VLA* — the agent never learned a control policy purely from its own reward. This lab closes that gap with **Proximal Policy Optimization (PPO)**, the workhorse on-policy RL algorithm. We build the actor/value networks, the clipped-surrogate update, and Generalized Advantage Estimation (GAE) from scratch. Following the MT05 pattern, the in-notebook training is a **short demo** (50k steps) that only shows the learning loop and its curves — 50k is far too few for `HalfCheetah`, so that policy is still weak. The polished **running gait** rollout then loads a **fully-trained (baked) 300k-step checkpoint**. (`2leg_cheetah` is wrapped so the episode ends if the torso flips past ~90°: HalfCheetah's reward doesn't penalize flipping, so without this PPO learns to paddle forward on its back — a high-reward but broken-looking gait.) This from-scratch on-policy baseline is the natural setup for **MT07**, which asks how a policy trained in one domain can be *transferred* to another.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + PyTorch environment (the `auplc-mujoco-torch` course image: torch + mujoco + robosuite + gymnasium) that this notebook is built for. The shared RL utilities (`cdrl_code/`) ship next to this notebook.

## Goals
- Understand the on-policy PPO pipeline: rollout collection → GAE → clipped update
- Build the actor (Gaussian policy) and value networks, and the PPO agent
- Run a **short (50k) in-notebook demo train** on `2leg_cheetah` and read its curves
- Understand why 50k is undertrained (the demo policy still flails)
- Load the **fully-trained (baked) 300k checkpoint** and render the learned running gait
- (Appendix) reproduce that 300k checkpoint from scratch

### Setup

We force the EGL GL backend (the reliable offscreen path on this ROCm/gfx1151 stack), put `cdrl_code/` on `sys.path`, and import the shared RL helpers: `PPOBuffer` (the on-policy rollout buffer), `construct_env` (environment factory), `EvalManager` (periodic evaluation), and `seed_everything`.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import sys
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
import imageio
import mujoco
from argparse import Namespace
from tqdm import tqdm
from PIL import Image, ImageDraw
from IPython.display import Video

import logging
logging.getLogger("robosuite_logs").setLevel(logging.ERROR)

# The shared RL utilities ship next to this notebook in cdrl_code/; the baked
# fully-trained checkpoint lives in cdrl_assets/ (same as MT07; MT05's baked
# checkpoint instead lives in the image's Hugging Face cache).
CDRL_CODE = os.path.abspath("cdrl_code")
CDRL_ASSETS = os.path.abspath("cdrl_assets")
sys.path.insert(0, CDRL_CODE)
from utils import PPOBuffer, seed_everything, EvalManager, construct_env  # noqa: E402

os.makedirs("output/videos", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__, "| mujoco", mujoco.__version__)

### Actor and value networks

PPO uses two networks. The **actor** outputs the mean of a Gaussian policy over continuous actions (the log-standard-deviation is a separate state-independent parameter); the **value** network estimates the expected return of a state. Orthogonal initialization (`layer_init`) with a small final-layer gain is the standard PPO trick for stable early training.

In [ ]:
def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer

def build_actor(obs_dim, act_dim, hidden_size, activation):
    return nn.Sequential(
        layer_init(nn.Linear(obs_dim, hidden_size)),
        activation(),
        layer_init(nn.Linear(hidden_size, hidden_size)),
        activation(),
        layer_init(nn.Linear(hidden_size, act_dim), std=0.01),
    )


def build_value_function(obs_dim, hidden_size, activation):
    return nn.Sequential(
        layer_init(nn.Linear(obs_dim, hidden_size)),
        activation(),
        layer_init(nn.Linear(hidden_size, hidden_size)),
        activation(),
        layer_init(nn.Linear(hidden_size, 1), std=1.0),
    )

### Generalized Advantage Estimation (GAE)

Before the policy update we turn the rollout's rewards and value estimates into **advantages** (how much better an action was than the value baseline) and **returns** (the value-function targets). GAE walks *backward* through the rollout, accumulating the temporal-difference residual

\[ \delta_t = r_t + \gamma\, V(s_{t+1})\,(1-\text{done}_t) - V(s_t) \]

with an exponential `gae_lambda` weighting to trade off bias vs. variance. Returns are then just `advantages + values`.

In [ ]:
def compute_gae(rewards, values, next_obs_values, dones, gamma, gae_lambda):
    """Generalized Advantage Estimation over one rollout.

    Walks backward accumulating the TD residual
    delta_t = r_t + gamma * V(s_{t+1}) * (1 - done_t) - V(s_t)
    into the GAE advantage, then forms returns = advantages + values.
    """
    advantages = torch.zeros_like(rewards)
    lastgaelam = 0.0
    num_steps = rewards.shape[0]
    for t in reversed(range(num_steps)):
        delta = rewards[t] + gamma * next_obs_values[t] * (1.0 - dones[t]) - values[t]
        advantages[t] = lastgaelam = delta + gamma * gae_lambda * (1.0 - dones[t]) * lastgaelam
    returns = advantages + values
    return advantages, returns

### The clipped-surrogate PPO update

`ppo_update` runs several epochs of minibatch SGD over the collected rollout. For each minibatch it recomputes the new log-probabilities, forms the probability `ratio` against the old policy, and optimizes the **clipped surrogate objective** (so a single update cannot move the policy too far), plus a (optionally clipped) value-function loss and an optional entropy bonus. It tracks the approximate KL and clip fraction, applies global gradient-norm clipping, and can early-stop when the approximate KL exceeds `target_kl`.

In [ ]:
def ppo_update(agent, advantages, returns):
    """Run the clipped-surrogate PPO optimization on `agent` for one rollout.

    Returns the last-minibatch loss tensors and diagnostics the caller needs to
    log (policy/value/entropy losses, approx KL, clip fractions, and the flat
    values/returns used for explained-variance)."""
    cfg = agent.training_config

    b_obs = agent.buffer.obs.reshape((-1, agent.obs_dim))
    b_logprobs = agent.buffer.logprobs.reshape(-1)
    b_actions = agent.buffer.actions.reshape((-1, agent.act_dim))
    b_advantages = advantages.reshape(-1)
    b_returns = returns.reshape(-1)
    b_values = agent.buffer.values.reshape(-1)

    b_inds = np.arange(agent.buffer.num_steps)
    clipfracs = []

    for epoch in range(cfg["update_epochs"]):
        np.random.shuffle(b_inds)

        for start in range(0, agent.buffer.num_steps, cfg["minibatch_size"]):
            end = start + cfg["minibatch_size"]
            mb_inds = b_inds[start:end]

            _, newlogprob, entropy, newvalue = agent.get_action_and_value(
                b_obs[mb_inds],
                b_actions[mb_inds],
            )

            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                old_approx_kl = (-logratio).mean()
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs.append(((ratio - 1.0).abs() > cfg["clip_coef"]).float().mean().item())

            mb_advantages = b_advantages[mb_inds]
            if cfg["norm_adv"]:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            pg_loss1 = -mb_advantages * ratio
            pg_loss2 = -mb_advantages * torch.clamp(ratio, 1 - cfg["clip_coef"], 1 + cfg["clip_coef"])
            pg_loss = torch.max(pg_loss1, pg_loss2).mean()

            newvalue = newvalue.view(-1)
            if cfg["clip_vloss"] is not None:
                v_loss_unclipped = (newvalue - b_returns[mb_inds]) ** 2
                v_clipped = b_values[mb_inds] + torch.clamp(
                    newvalue - b_values[mb_inds],
                    -cfg["clip_vloss"],
                    cfg["clip_vloss"],
                )
                v_loss_clipped = (v_clipped - b_returns[mb_inds]) ** 2
                v_loss = 0.5 * torch.max(v_loss_unclipped, v_loss_clipped).mean()
            else:
                v_loss = 0.5 * ((newvalue - b_returns[mb_inds]) ** 2).mean()

            entropy_loss = entropy.mean()
            loss = pg_loss - cfg["ent_coef"] * entropy_loss + cfg["vf_coef"] * v_loss

            agent.optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.params, cfg["max_grad_norm"])
            agent.optimizer.step()

        if cfg["target_kl"] is not None and approx_kl > cfg["target_kl"]:
            break

    return {
        "pg_loss": pg_loss,
        "v_loss": v_loss,
        "entropy_loss": entropy_loss,
        "old_approx_kl": old_approx_kl,
        "approx_kl": approx_kl,
        "clipfracs": clipfracs,
        "b_values": b_values,
        "b_returns": b_returns,
    }

### The PPO agent

The agent bundles the actor, the value function, the rollout buffer, the optimizer, action sampling, and checkpoint save/load. It delegates the two heavy pieces to the module-level `compute_gae` and `ppo_update` defined above: `_compute_gae_and_returns` calls the former, and `train` calls the latter (which maximizes the **clipped surrogate objective** with GAE advantages, adds a value-function loss and an optional entropy bonus, and can early-stop an epoch when the approximate KL exceeds `target_kl`).

In [ ]:
class PPO(object):
    def __init__(self, args, obs_dim, act_dim, nn_act, device):
        self.interact = "ON_POLICY"
        self.device = device
        self.obs_dim = obs_dim
        self.act_dim = act_dim
        self.hidden_size = args.nn_hidden_size
        self.anneal_lr = args.anneal_lr

        self.mu_net = build_actor(obs_dim, act_dim, args.nn_hidden_size, nn_act).to(device)
        self.V = build_value_function(obs_dim, args.nn_hidden_size, nn_act).to(device)

        # State-independent log standard deviation for the Gaussian policy.
        self.log_std = nn.Parameter(torch.zeros(act_dim, device=device))

        self.params = list(self.mu_net.parameters()) + list(self.V.parameters()) + [self.log_std]
        self.optimizer = optim.Adam(self.params, lr=args.learning_rate, eps=1e-5)

        self.buffer = PPOBuffer(args.num_steps, obs_dim, act_dim, device)

        self.training_config = {
            "learning_rate": args.learning_rate,
            "update_epochs": args.update_epochs,
            "gamma": args.gamma,
            "gae_lambda": args.gae_lambda,
            "minibatch_size": int(args.num_steps // args.num_minibatches),
            "norm_adv": args.norm_adv,
            "clip_coef": args.clip_coef,
            "clip_vloss": args.clip_vloss,
            "ent_coef": args.ent_coef,
            "vf_coef": args.vf_coef,
            "max_grad_norm": args.max_grad_norm,
            "target_kl": args.target_kl,
        }

        self.training_history = {
            "learning_rate": [],
            "value_loss": [],
            "policy_loss": [],
            "entropy": [],
            "old_approx_kl": [],
            "approx_kl": [],
            "clipfrac": [],
            "explained_variance": [],
            "SPS": [],
        }

        self.start_time = time.time()

    def anneal_anything(self, iteration, num_iterations):
        if self.anneal_lr:
            frac = 1.0 - (iteration - 1.0) / num_iterations
            lrnow = frac * self.training_config["learning_rate"]
            self.optimizer.param_groups[0]["lr"] = lrnow

    def get_action_and_value(self, obs, action=None):
        obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        mu = self.mu_net(obs)
        std = torch.exp(self.log_std)
        probs = Normal(mu, std)

        if action is None:
            action = probs.sample()
        else:
            action = torch.as_tensor(action, dtype=torch.float32, device=self.device)

        logprob = probs.log_prob(action).sum(dim=-1)
        entropy = probs.entropy().sum(dim=-1)
        value = self.V(obs)
        return action, logprob, entropy, value

    def get_action(self, obs, test=False):
        obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        mu = self.mu_net(obs)

        if test:
            return mu

        std = torch.exp(self.log_std)
        return Normal(mu, std).sample()

    def get_value(self, obs):
        with torch.no_grad():
            obs = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
            return self.V(obs)

    def store_data(self, step, obs, action, next_obs, reward, done, value, next_obs_value, logprob):
        reward = torch.tensor(reward, dtype=torch.float32, device=self.device).view(-1)
        done = torch.tensor(done, dtype=torch.float32, device=self.device).view(-1)

        self.buffer.store(
            step,
            obs[0],
            action[0],
            next_obs[0],
            reward,
            done,
            value.flatten(),
            next_obs_value.flatten(),
            logprob.flatten(),
        )

    def save(self, path):
        checkpoint = {
            "mu_net_state_dict": self.mu_net.state_dict(),
            "V_state_dict": self.V.state_dict(),
            "log_std": self.log_std.detach().cpu(),
            "training_config": self.training_config,
        }
        torch.save(checkpoint, path)
        print("Model saved to:", path)

    def load(self, path, device=None):
        if device is None:
            device = self.device

        checkpoint = torch.load(path, map_location=device)
        self.mu_net.load_state_dict(checkpoint["mu_net_state_dict"])
        self.V.load_state_dict(checkpoint["V_state_dict"])

        with torch.no_grad():
            self.log_std.copy_(checkpoint["log_std"].to(device))

        self.mu_net.to(device)
        self.V.to(device)
        self.log_std.data = self.log_std.data.to(device)
        self.device = device
        print("Model loaded from:", path)

    def _compute_gae_and_returns(self):
        with torch.no_grad():
            advantages = torch.zeros_like(self.buffer.rewards, device=self.device)
            lastgaelam = 0

            for t in reversed(range(self.buffer.num_steps)):
                delta = (
                    self.buffer.rewards[t]
                    + self.training_config["gamma"]
                    * self.buffer.next_obs_values[t]
                    * (1.0 - self.buffer.dones[t])
                    - self.buffer.values[t]
                )

                advantages[t] = lastgaelam = (
                    delta
                    + self.training_config["gamma"]
                    * self.training_config["gae_lambda"]
                    * (1.0 - self.buffer.dones[t])
                    * lastgaelam
                )

            returns = advantages + self.buffer.values

        return advantages, returns

    def train(self, step, writer=None):
        advantages, returns = self._compute_gae_and_returns()

        # The clipped-surrogate optimization lives in the module-level ppo_update
        # (defined above) so it can be taught on its own.
        out = ppo_update(self, advantages, returns)
        pg_loss = out["pg_loss"]
        v_loss = out["v_loss"]
        entropy_loss = out["entropy_loss"]
        old_approx_kl = out["old_approx_kl"]
        approx_kl = out["approx_kl"]
        clipfracs = out["clipfracs"]
        b_values = out["b_values"]
        b_returns = out["b_returns"]

        y_pred = b_values.detach().cpu().numpy()
        y_true = b_returns.detach().cpu().numpy()
        var_y = np.var(y_true)
        explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y
        sps = int(step / (time.time() - self.start_time))

        metrics = {
            "learning_rate": self.optimizer.param_groups[0]["lr"],
            "value_loss": v_loss.item(),
            "policy_loss": pg_loss.item(),
            "entropy": entropy_loss.item(),
            "old_approx_kl": old_approx_kl.item(),
            "approx_kl": approx_kl.item(),
            "clipfrac": np.mean(clipfracs),
            "explained_variance": explained_var,
            "SPS": sps,
        }

        for key, value in metrics.items():
            self.training_history[key].append(value)

        if writer is not None:
            writer.add_scalar("charts/learning_rate", metrics["learning_rate"], step)
            writer.add_scalar("losses/value_loss", metrics["value_loss"], step)
            writer.add_scalar("losses/policy_loss", metrics["policy_loss"], step)
            writer.add_scalar("losses/entropy", metrics["entropy"], step)
            writer.add_scalar("losses/old_approx_kl", metrics["old_approx_kl"], step)
            writer.add_scalar("losses/approx_kl", metrics["approx_kl"], step)
            writer.add_scalar("losses/clipfrac", metrics["clipfrac"], step)
            writer.add_scalar("losses/explained_variance", metrics["explained_variance"], step)
            writer.add_scalar("charts/SPS", metrics["SPS"], step)

        return metrics

### Training configuration

`env_name="2leg_cheetah"` is Gymnasium's `HalfCheetah-v5`. Following the MT05 pattern, the in-notebook training is a **short 50k-step demo** (`total_timesteps=50000`, rollouts of `num_steps=2048`) — enough to see the learning curves, but far too short for a real gait. The rollout later loads a **baked 300k-step checkpoint** (`PPO_CKPT`) for the polished result; `FULL_TIMESTEPS` is the budget the appendix uses to reproduce it.

In [ ]:
args_dict = {
    "seed": 0,
    "total_timesteps": 50000,
    "num_steps": 2048,
    "learning_rate": 0.0003,
    "nn_hidden_size": 64,
    "num_minibatches": 32,
    "update_epochs": 10,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "anneal_lr": True,
    "norm_adv": True,
    "clip_coef": 0.2,
    "clip_vloss": 0.2,
    "ent_coef": 0.0,
    "vf_coef": 0.5,
    "max_grad_norm": 0.5,
    "target_kl": None,
    "evaluate_freq": 10000,
    "save_model_interval": [],
    "cuda": True,
    "env_name": "2leg_cheetah",
}
args = Namespace(**args_dict)

# Budget + path for the fully-trained checkpoint used by the rollout (baked in cdrl_assets/,
# same pattern as MT05's SmolVLA checkpoint). The appendix reproduces it.
FULL_TIMESTEPS = 300000
PPO_CKPT = os.path.join(CDRL_ASSETS, "checkpoints", "ppo_cheetah", "steps_300000.pt")

log_path = f"output/logs/PPO/{args.env_name}/{args.seed}"
os.makedirs(os.path.join(log_path, "models"), exist_ok=True)
print("demo steps:", args.total_timesteps, "| baked checkpoint:", PPO_CKPT)

### Build environment and initialize PPO

Construct the environment and create the PPO agent sized to its observation/action dimensions.

In [ ]:
envs = construct_env(args.env_name, args.seed)[0]

agent = PPO(
    args=args,
    obs_dim=envs.observation_space.shape[0],
    act_dim=envs.action_space.shape[0],
    nn_act=nn.Tanh,
    device=device,
)
print("obs/act:", envs.observation_space.shape, envs.action_space.shape)

### On-policy RL pipeline: collect → GAE → update

The loop follows the standard on-policy workflow: collect a rollout with the current policy, compute advantages/returns with GAE, run several epochs of minibatch PPO updates, and evaluate. This is the **short 50k-step demo** — enough to watch the loss/reward curves move, but not a finished policy. The polished gait comes from the baked 300k checkpoint two cells down.

### Collecting an on-policy rollout

PPO is **on-policy**: each update uses a fresh batch of transitions gathered by the *current* policy. `collect_rollout` fills the buffer with exactly `num_steps` transitions — sampling an action and value, stepping the env, bootstrapping the next-state value, and storing everything — resetting whenever an episode ends. It returns the running observation so the next iteration continues seamlessly.

In [ ]:
def collect_rollout(agent, envs, obs, args):
    """Gather one on-policy rollout of `args.num_steps` transitions into
    `agent.buffer`. Returns the observation to carry into the next iteration."""
    for rollout_step in range(args.num_steps):
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(obs)

        action_np = np.clip(action[0].cpu().numpy(), envs.action_space.low, envs.action_space.high)
        next_obs, reward, termination, truncation, _ = envs.step(action_np)
        done = termination or truncation
        next_obs_tensor = torch.tensor(next_obs, dtype=torch.float32, device=agent.device).unsqueeze(0)
        next_obs_value = agent.get_value(next_obs_tensor)

        agent.store_data(
            rollout_step, obs, action, next_obs_tensor,
            reward, done, value, next_obs_value, logprob,
        )

        if done:
            next_obs, _ = envs.reset()
            next_obs_tensor = torch.tensor(next_obs, dtype=torch.float32, device=agent.device).unsqueeze(0)

        obs = next_obs_tensor
    return obs

In [ ]:
def run_training(agent, args, envs, log_path):
    seed_everything(args.seed)
    envs.action_space.seed(args.seed)
    eval_env = EvalManager(construct_env(args.env_name, args.seed)[0], device=device, reset_seed=args.seed, eval_episodes=5)
    obs, _ = envs.reset(seed=args.seed)
    obs = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

    num_iterations = args.total_timesteps // args.num_steps
    eval_results = {}
    save_steps_idx = 0
    next_eval = args.evaluate_freq

    pbar = tqdm(range(1, num_iterations + 1), desc="PPO", ncols=100)
    for iteration in pbar:
        agent.anneal_anything(iteration, num_iterations)

        # Collect one on-policy rollout of num_steps transitions (see
        # collect_rollout above); it returns the observation to carry forward.
        obs = collect_rollout(agent, envs, obs, args)

        global_step = iteration * args.num_steps
        agent.train(global_step)
        # Evaluate only every `evaluate_freq` steps (plus the final iteration),
        # quietly (verbose=False), and surface the latest reward on the tqdm bar
        # itself via set_postfix. This keeps the output to a single clean line
        # instead of one progress bar + one [Eval] line per iteration.
        if global_step >= next_eval or iteration == num_iterations:
            eval_rews, eval_lens, success = eval_env.evaluate(agent, global_step, verbose=False)
            eval_results[global_step] = {
                "rewards": eval_rews,
                "lengths": eval_lens,
                "success": success,
            }
            pbar.set_postfix(eval_rew=f"{eval_rews:.0f}")
            next_eval += args.evaluate_freq

        if save_steps_idx < len(args.save_model_interval) and global_step > args.save_model_interval[save_steps_idx]:
            ckpt_path = os.path.join(log_path, "models", f"steps_{global_step}.pt")
            agent.save(ckpt_path)
            save_steps_idx += 1

    envs.close()
    agent.save(os.path.join(log_path, "models", f"steps_final.pt"))
    return eval_results


# Run training.
eval_results = run_training(agent, args, envs, log_path)

### Training curves

We visualize the main PPO training signals — policy loss, value loss, entropy, approximate KL divergence, clip fraction, and the periodic evaluation reward. A healthy run shows the evaluation reward trending up while the approximate KL and clip fraction stay in a sane range.

In [ ]:
import matplotlib.pyplot as plt


def plot_ppo_curves(agent, eval_results=None):
    plt.figure(figsize=(16, 10))

    plt.subplot(2, 3, 1)
    plt.plot(agent.training_history["policy_loss"], label="PPO")
    plt.title("Policy Loss")
    plt.xlabel("Update")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(2, 3, 2)
    plt.plot(agent.training_history["value_loss"], label="PPO")
    plt.title("Value Loss")
    plt.xlabel("Update")
    plt.ylabel("Loss")
    plt.legend()

    plt.subplot(2, 3, 3)
    plt.plot(agent.training_history["entropy"], label="PPO")
    plt.title("Entropy")
    plt.xlabel("Update")
    plt.ylabel("Entropy")
    plt.legend()

    plt.subplot(2, 3, 4)
    plt.plot(agent.training_history["approx_kl"], label="PPO")
    plt.title("Approximate KL")
    plt.xlabel("Update")
    plt.ylabel("KL")
    plt.legend()

    plt.subplot(2, 3, 5)
    plt.plot(agent.training_history["clipfrac"], label="PPO")
    plt.title("Clip Fraction")
    plt.xlabel("Update")
    plt.ylabel("Clip Fraction")
    plt.legend()

    if eval_results:
        eval_steps = sorted(eval_results.keys())
        eval_rewards = [np.mean(eval_results[s]["rewards"]) for s in eval_steps]

        plt.subplot(2, 3, 6)
        plt.plot(eval_steps, eval_rewards, marker="o", label="PPO")
        plt.title("Evaluation Reward")
        plt.xlabel("Environment Step")
        plt.ylabel("Average Reward")
        plt.legend()

    plt.tight_layout()
    plt.show()


plot_ppo_curves(agent, eval_results)

### Rollout: the learned gait (fully-trained 300k checkpoint)

The 50k demo above only *starts* learning. Here we load the **baked 300k-step checkpoint** (`PPO_CKPT`, trained the same way for 6× longer) and render a rollout of the deterministic policy (the Gaussian mean), writing an `.mp4` to `output/videos/` — this is where you see a real upright running gait rather than the demo's weak motion. If the checkpoint is missing, run the appendix once to produce it.

In [ ]:
def rollout_video(agent, args, out_path, max_steps=1000, fps=30):
    env = construct_env(args.env_name, args.seed, render_mode="rgb_array")[0]
    obs, _ = env.reset(seed=args.seed)
    obs = torch.as_tensor(obs, dtype=torch.float32, device=device)
    frames, total_reward = [], 0.0
    for step in tqdm(range(max_steps)):
        with torch.no_grad():
            action = agent.get_action(obs, test=True)
        action_np = np.clip(action.cpu().numpy(), env.action_space.low, env.action_space.high)
        next_obs, reward, term, trunc, _ = env.step(action_np)
        total_reward += float(reward)
        frame = env.render()
        img = Image.fromarray(frame)
        ImageDraw.Draw(img).text((12, 10), f"PPO  step={step}  return={total_reward:.1f}", fill=(255, 255, 255))
        frames.append(np.array(img))
        if term or trunc:
            break
        obs = torch.as_tensor(next_obs, dtype=torch.float32, device=device)
    env.close()
    imageio.mimsave(out_path, frames, fps=fps)
    print(f"saved {out_path} | return={total_reward:.1f} | frames={len(frames)}")
    return out_path


if os.path.exists(PPO_CKPT):
    agent.load(PPO_CKPT)
    rollout_video(agent, args, "output/videos/mt06_ppo_cheetah.mp4", max_steps=240)
else:
    print(f"Baked checkpoint not found: {PPO_CKPT}")
    print("Run the appendix cell (train_full_ppo(PPO_CKPT)) once to produce it, "
          "then re-run this cell.")

In [ ]:
Video(url="output/videos/mt06_ppo_cheetah.mp4")

## Appendix: reproduce the fully-trained 300k checkpoint

The rollout above uses a baked `PPO_CKPT`. This appendix reproduces it from scratch
with the **same code and hyperparameters**, just trained for `FULL_TIMESTEPS=300000`
steps instead of the 50k demo, and saves it to `PPO_CKPT`. It is **not** part of the
normal top-to-bottom pass (it takes ~5 min on the GPU) — call `train_full_ppo(PPO_CKPT)`
once to (re)generate the checkpoint, then re-run the rollout cell. After regenerating,
rebuild the course image so the new checkpoint is baked in.

In [ ]:
def train_full_ppo(save_path, timesteps=FULL_TIMESTEPS):
    """Train PPO from scratch for `timesteps` and save the checkpoint to `save_path`.
    Run once offline to (re)generate the baked checkpoint used by the rollout."""
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    full_log = "output/logs/PPO/full"
    os.makedirs(os.path.join(full_log, "models"), exist_ok=True)
    full_args = Namespace(**{**args_dict, "total_timesteps": timesteps, "save_model_interval": []})
    env = construct_env(full_args.env_name, full_args.seed)[0]
    full_agent = PPO(full_args, env.observation_space.shape[0], env.action_space.shape[0],
                     nn.Tanh, device)
    run_training(full_agent, full_args, env, full_log)
    full_agent.save(save_path)
    print("baked checkpoint saved to", save_path)
    return full_agent


# Uncomment to (re)train the baked 300k checkpoint (~5 min on GPU):
# train_full_ppo(PPO_CKPT)

## Conclusions

You implemented **PPO** end to end — the Gaussian actor and value networks, a fixed-size on-policy rollout buffer, Generalized Advantage Estimation, and the clipped-surrogate update with an optional KL early-stop. The short 50k demo showed the on-policy learning signal (and why so few steps is undertrained), while the **baked 300k checkpoint** rollout shows the emergent upright running gait, learned purely from reward with no demonstrations or pretrained weights. This from-scratch on-policy baseline sets up **MT07**, where cross-domain RL (QAvatar) reuses knowledge from a related source domain to learn a new target domain faster than training from zero.

## Acknowledgements

This notebook was developed in collaboration with the lab of Professor Ping-Chun Hsieh (謝秉均), Department of Computer Science, National Yang Ming Chiao Tung University (NYCU) — [pinghsieh.github.io](https://pinghsieh.github.io/).

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT